In [ ]:
# ISIC 2019 Ablation — Config 1: Metadata Only (Tabular MLP)

import os, random, warnings, logging
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

import numpy as np
import pandas as pd
from collections import Counter
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.amp import autocast, GradScaler

from torchinfo import summary
from tqdm import tqdm
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, accuracy_score
)
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

In [ ]:
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64   # larger batch fine since no image loading

train_dir = "/kaggle/input/datasets/sujansarkar/isic-2019-multi-modal-classification/Train_Validation_dataset/content/Train_Validation_dataset/train"
val_dir   = "/kaggle/input/datasets/sujansarkar/isic-2019-multi-modal-classification/Train_Validation_dataset/content/Train_Validation_dataset/val"
test_dir  = "/kaggle/input/datasets/sujansarkar/isic-2019-multi-modal-classification/Test_dataset/content/test_dataset2"
meta_path = "/kaggle/input/datasets/sujansarkar/isic-2019-multi-modal-classification/ISIC_2019_Training_Metadata.csv"

In [ ]:
from torch.utils.data import Dataset

df = pd.read_csv(meta_path)
if "image" in df.columns:
    df = df.set_index("image")

# Normalise age using train image set
train_imgs = set(
    os.path.splitext(os.path.basename(p))[0]
    for p, _ in datasets.ImageFolder(train_dir).samples
)
df["age_approx"] = pd.to_numeric(df["age_approx"], errors="coerce")
age_mean = df.loc[df.index.isin(train_imgs), "age_approx"].mean()
age_std  = df.loc[df.index.isin(train_imgs), "age_approx"].std() + 1e-8
df["age_norm"] = (df["age_approx"].fillna(age_mean) - age_mean) / age_std

for col in ["sex", "anatom_site_general"]:
    df[col] = df[col].fillna("unknown")
df = pd.get_dummies(df, columns=["sex","anatom_site_general"],
                    dummy_na=False, dtype=float)

TABULAR_COLS = ["age_norm"] + [c for c in df.columns
                if c.startswith("sex_") or c.startswith("anatom_site_general_")]
NUM_TAB = len(TABULAR_COLS)
print(f"Tabular features: {NUM_TAB} — {TABULAR_COLS}")


class MetadataOnlyDataset(Dataset):
    """
    Uses ImageFolder to discover split membership and labels,
    but returns only tabular features — no images loaded.
    Falls back to zero vector if image has no metadata entry.
    """
    def __init__(self, root_dir, df, tab_cols):
        folder          = datasets.ImageFolder(root_dir)
        self.samples    = folder.samples   # (path, label_idx)
        self.class_to_idx = folder.class_to_idx
        self.df         = df
        self.tab_cols   = tab_cols
        self._zero      = torch.zeros(len(tab_cols))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img_name    = os.path.splitext(os.path.basename(path))[0]
        if img_name in self.df.index:
            tab = torch.tensor(
                self.df.loc[img_name, self.tab_cols].values.astype(float),
                dtype=torch.float32
            )
        else:
            tab = self._zero
        return tab, label

train_dataset = MetadataOnlyDataset(train_dir, df, TABULAR_COLS)
val_dataset   = MetadataOnlyDataset(val_dir,   df, TABULAR_COLS)
test_dataset  = MetadataOnlyDataset(test_dir,  df, TABULAR_COLS)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
class_to_idx = train_dataset.class_to_idx
IDX_TO_CLASS = {
    class_to_idx["AK"]:   "Actinic Keratoses (AK)",
    class_to_idx["BCC"]:  "Basal Cell Carcinoma (BCC)",
    class_to_idx["BKL"]:  "Benign Keratosis-like Lesions (BKL)",
    class_to_idx["DF"]:   "Dermatofibroma (DF)",
    class_to_idx["MEL"]:  "Melanoma (MEL)",
    class_to_idx["NV"]:   "Melanocytic Nevi (NV)",
    class_to_idx["SCC"]:  "Squamous Cell Carcinoma (SCC)",
    class_to_idx["VASC"]: "Vascular Lesions (VASC)",
}
NUM_CLASSES = len(IDX_TO_CLASS)
class_names = [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)]
for i, n in enumerate(class_names): print(i, "→", n)

In [ ]:
class MetadataOnlyNet(nn.Module):
    """
    Three-layer MLP on tabular metadata only — no image, no text.
    Establishes the discriminative ceiling of demographics alone.
    """
    def __init__(self, num_meta_features, num_classes=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(num_meta_features, 256),
            nn.BatchNorm1d(256), nn.ReLU(inplace=True), nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True), nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

    def forward(self, tab):
        return self.net(tab)

model = MetadataOnlyNet(num_meta_features=NUM_TAB, num_classes=NUM_CLASSES).to(DEVICE)
summary(model, input_size=(BATCH_SIZE, NUM_TAB),
        col_names=["input_size","output_size","num_params"], depth=3)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.ce    = nn.CrossEntropyLoss(reduction="none", label_smoothing=0.1)
    def forward(self, inputs, targets):
        ce = self.ce(inputs, targets)
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()

In [ ]:
def compute_specificity(y_true, y_pred, n):
    cm = confusion_matrix(y_true, y_pred, labels=range(n))
    s = []
    for i in range(n):
        TP=cm[i,i]; FP=cm[:,i].sum()-TP; FN=cm[i,:].sum()-TP; TN=cm.sum()-(TP+FP+FN)
        s.append(TN/(TN+FP) if (TN+FP)>0 else 0)
    return np.mean(s)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss=0.0; yt,yp=[],[]
    for tab, labels in tqdm(loader, desc="Train", leave=False):
        tab=tab.to(device); labels=labels.to(device)
        optimizer.zero_grad()
        out  = model(tab)
        loss = criterion(out, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item()*tab.size(0)
        yt.extend(labels.cpu().numpy()); yp.extend(out.argmax(1).cpu().numpy())
    L=running_loss/len(yt)
    return (L, accuracy_score(yt,yp),
            precision_score(yt,yp,average="macro",zero_division=0),
            recall_score(yt,yp,average="macro",zero_division=0),
            f1_score(yt,yp,average="macro",zero_division=0),
            compute_specificity(yt,yp,NUM_CLASSES))

def evaluate(model, loader, criterion, device):
    model.eval(); running_loss=0.0; yt,yp=[],[]
    with torch.no_grad():
        for tab, labels in tqdm(loader, desc="Eval", leave=False):
            tab=tab.to(device); labels=labels.to(device)
            out=model(tab); loss=criterion(out,labels)
            running_loss+=loss.item()*tab.size(0)
            yt.extend(labels.cpu().numpy()); yp.extend(out.argmax(1).cpu().numpy())
    L=running_loss/len(yt)
    return (L, accuracy_score(yt,yp),
            precision_score(yt,yp,average="macro",zero_division=0),
            recall_score(yt,yp,average="macro",zero_division=0),
            f1_score(yt,yp,average="macro",zero_division=0),
            compute_specificity(yt,yp,NUM_CLASSES), yt, yp)

In [ ]:
criterion = FocalLoss(gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=2, eta_min=1e-6
)

EPOCHS = 60
RESTART_EPOCHS = {10, 30}
best_val_f1=0.0; best_model_state=None; patience=15; counter=0

for epoch in range(EPOCHS):
    print(f"\nEpoch [{epoch+1}/{EPOCHS}]")
    if (epoch+1) in RESTART_EPOCHS: counter=0; print("LR restart — patience reset")
    tr = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    vl = evaluate(model, val_loader, criterion, DEVICE)
    scheduler.step()
    print(f"LR: {optimizer.param_groups[0]['lr']:.2e}")
    print(f"Train | Loss:{tr[0]:.4f} Acc:{tr[1]:.4f} F1:{tr[4]:.4f} Spec:{tr[5]:.4f}")
    print(f"Val   | Loss:{vl[0]:.4f} Acc:{vl[1]:.4f} F1:{vl[4]:.4f} Spec:{vl[5]:.4f}")
    if vl[4] > best_val_f1:
        best_val_f1=vl[4]; best_model_state=model.state_dict()
        torch.save(best_model_state, "/kaggle/working/ISIC2019_C1_MetadataOnly_Best.pth")
        counter=0; print(f"Val F1 improved ({best_val_f1:.4f}) — saving")
    else:
        counter+=1
        if counter>=patience: print(f"Early stopping at epoch {epoch+1}"); break

if best_model_state: model.load_state_dict(best_model_state)
print(f"Done. Best val F1: {best_val_f1:.4f}")

In [ ]:
def print_report(split, model, loader, criterion):
    L,acc,prec,rec,f1,spec,yt,yp = evaluate(model, loader, criterion, DEVICE)
    print(f"\n{'='*50}\n{split.upper()} REPORT")
    print(f"Loss:{L:.4f} | Acc:{acc:.4f} | Spec:{spec:.4f}")
    print(classification_report(yt, yp, target_names=class_names, zero_division=0, digits=4))
    mprec=precision_score(yt,yp,average="macro",zero_division=0)
    mrec=recall_score(yt,yp,average="macro",zero_division=0)
    mf1=f1_score(yt,yp,average="macro",zero_division=0)
    print(f"Macro → Prec:{mprec:.4f} | Rec:{mrec:.4f} | F1:{mf1:.4f} | Spec:{spec:.4f}")
    return yt, yp

train_yt,train_yp = print_report("Train",      model, train_loader, criterion)
val_yt,val_yp     = print_report("Validation", model, val_loader,   criterion)
test_yt,test_yp   = print_report("Test",       model, test_loader,  criterion)

In [ ]:
from scipy import stats as scipy_stats

CONFIG = "C1"

N_BOOTSTRAP = 1000
rng         = np.random.default_rng(42)
y_true_arr  = np.array(test_yt)
y_pred_arr  = np.array(test_yp)
n_test      = len(y_true_arr)

boot = {"Accuracy": [], "F1 (Macro)": [], "Precision (Macro)": [], "Recall (Macro)": [], "Specificity": []}
per_class_f1 = {name: [] for name in class_names}

def compute_specificity_boot(y_t, y_p, num_classes):
    cm = confusion_matrix(y_t, y_p, labels=list(range(num_classes)))
    specs = []
    for i in range(num_classes):
        TP = cm[i, i]; FP = cm[:, i].sum() - TP
        FN = cm[i, :].sum() - TP; TN = cm.sum() - (TP + FP + FN)
        specs.append(TN / (TN + FP) if (TN + FP) > 0 else 0.0)
    return float(np.mean(specs))

for _ in range(N_BOOTSTRAP):
    idx = rng.integers(0, n_test, size=n_test)
    y_t, y_p = y_true_arr[idx], y_pred_arr[idx]
    boot["Accuracy"].append(accuracy_score(y_t, y_p))
    boot["F1 (Macro)"].append(f1_score(y_t, y_p, average="macro", zero_division=0))
    boot["Precision (Macro)"].append(precision_score(y_t, y_p, average="macro", zero_division=0))
    boot["Recall (Macro)"].append(recall_score(y_t, y_p, average="macro", zero_division=0))
    boot["Specificity"].append(compute_specificity_boot(y_t, y_p, NUM_CLASSES))
    pc = f1_score(y_t, y_p, average=None, zero_division=0, labels=list(range(NUM_CLASSES)))
    for ci, name in enumerate(class_names):
        per_class_f1[name].append(pc[ci])

def ci95(arr): return float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5))

short_names = [n.split("(")[-1].replace(")", "").strip() if "(" in n else n
               for n in class_names]

W = 70
print("\n" + "=" * W)
print(f"  ISIC 2019 — {CONFIG} — Bootstrap CI (n={N_BOOTSTRAP})")
print("=" * W)
print(f"{'Metric':<22} {'Mean':>7}  {'Std':>7}  {'CI Lower':>10}  {'CI Upper':>10}")
print("=" * W)
for metric, values in boot.items():
    arr = np.array(values); lo, hi = ci95(arr)
    print(f"{metric:<22} {arr.mean():>7.4f}  {arr.std():>7.4f}  {lo:>10.4f}  {hi:>10.4f}")
print("-" * W)
print(f"\n{'Class':<12} {'Mean':>7}  {'Std':>7}  {'CI Lower':>10}  {'CI Upper':>10}")
print("-" * W)
for sname, name in zip(short_names, class_names):
    arr = np.array(per_class_f1[name]); lo, hi = ci95(arr)
    print(f"{sname:<12} {arr.mean():>7.4f}  {arr.std():>7.4f}  {lo:>10.4f}  {hi:>10.4f}")
print("=" * W)

fig, axes = plt.subplots(1, 5, figsize=(18, 6))
fig.subplots_adjust(top=0.82, wspace=0.4)
for ax, (label, values) in zip(axes, boot.items()):
    arr = np.array(values); lo, hi = ci95(arr); mn, sd = arr.mean(), arr.std()
    ax.boxplot(arr, widths=0.5, patch_artist=True,
               boxprops=dict(facecolor="#4c72b0", alpha=0.75),
               medianprops=dict(color="white", linewidth=2),
               whiskerprops=dict(linewidth=1.2), capprops=dict(linewidth=1.2),
               flierprops=dict(marker="o", markersize=3, markerfacecolor="gray", alpha=0.5))
    ax.set_title(label, fontsize=10, fontweight="bold", pad=8)
    ax.set_xticks([]); ax.set_ylabel("Score", fontsize=9)
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    ax.text(1, ax.get_ylim()[1], f"μ={mn:.4f}\nσ={sd:.4f}\n95%CI [{lo:.4f},{hi:.4f}]",
            ha="center", va="top", fontsize=7.5, color="#222222",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", edgecolor="gray", alpha=0.8))
plt.suptitle(f"ISIC 2019 {CONFIG} — Bootstrap Metric Distributions (n={N_BOOTSTRAP}) — Test Set",
             fontsize=13, fontweight="bold", y=0.97)
plt.savefig(f"/kaggle/working/isic_{CONFIG.lower()}_bootstrap_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

means  = np.array([np.mean(per_class_f1[n]) for n in class_names])
stds   = np.array([np.std(per_class_f1[n])  for n in class_names])
lo_arr = np.array([ci95(per_class_f1[n])[0] for n in class_names])
hi_arr = np.array([ci95(per_class_f1[n])[1] for n in class_names])

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(NUM_CLASSES)
bars = ax.bar(x, means, yerr=[means - lo_arr, hi_arr - means], capsize=6,
              color="#4c72b0", alpha=0.8, ecolor="#333333",
              error_kw={"linewidth": 1.8, "capthick": 1.8}, width=0.55)
ax.set_xticks(x)
ax.set_xticklabels(short_names, fontsize=11)
ax.set_ylabel("F1 Score", fontsize=11); ax.set_ylim(0, 1.15)
ax.set_title(f"ISIC 2019 {CONFIG} — Per-Class F1 Mean ± 95% CI (Bootstrap)",
             fontsize=13, fontweight="bold")
ax.grid(axis="y", alpha=0.3, linestyle="--"); ax.spines[["top", "right"]].set_visible(False)
for bar, m, s, hi in zip(bars, means, stds, hi_arr):
    ax.text(bar.get_x() + bar.get_width() / 2, hi + 0.025,
            f"{m:.3f}\n±{s:.3f}", ha="center", va="bottom", fontsize=9, color="#111111")
plt.tight_layout()
plt.savefig(f"/kaggle/working/isic_{CONFIG.lower()}_bootstrap_perclass_f1.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
cm = confusion_matrix(test_yt, test_yp)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("Actual")
plt.title("ISIC 2019 Test Confusion Matrix — Config 1: Metadata Only")
plt.tight_layout(); plt.show()